# Sentiment Stability Analysis

This notebook loads the production sentiment stability feature set and yearly market returns, merges them by year, and evaluates whether sentiment deviation is related to forward returns.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd

BASE_DIR = get_project_root()
if str(BASE_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(BASE_DIR / 'src'))

STABILITY_PATH = BASE_DIR / 'features' / 'sentiment_stability.csv'
MARKET_PATH = BASE_DIR / 'features' / 'market_returns.csv'

print(f'Loading sentiment stability features from: {STABILITY_PATH}')
print(f'Loading market returns from: {MARKET_PATH}')

stability = pd.read_csv(STABILITY_PATH)
market_returns = pd.read_csv(MARKET_PATH)

stability['year'] = pd.to_numeric(stability['year'], errors='coerce').astype('Int64')
market_returns['year'] = pd.to_numeric(market_returns['year'], errors='coerce').astype('Int64')

stability = stability.sort_values(['company_id', 'year'], kind='mergesort').reset_index(drop=True)
market_returns = market_returns.sort_values('year', kind='mergesort').reset_index(drop=True)

print('stability rows:', len(stability))
print('market rows:', len(market_returns))
print('stability year range:', stability['year'].min(), '->', stability['year'].max())
print('market year range:', market_returns['year'].min(), '->', market_returns['year'].max())


In [ ]:
analysis_df = stability.merge(market_returns, on='year', how='left', validate='many_to_one')
analysis_df = analysis_df.dropna(subset=['sentiment_deviation', 'next_year_return']).copy()
analysis_df = (
    analysis_df
    .groupby('year', as_index=False)
    .agg(
        sentiment_deviation=('sentiment_deviation', 'mean'),
        next_year_return=('next_year_return', 'first')
    )
    .sort_values('year', kind='mergesort')
    .reset_index(drop=True)
)

unique_years = analysis_df['year'].nunique()
row_count = len(analysis_df)
duplicate_years = analysis_df.loc[analysis_df['year'].duplicated(), 'year'].tolist()

print('merged rows after yearly aggregation:', row_count)
print('unique years count:', unique_years)
print('confirm 1 row per year:', row_count == unique_years)

if duplicate_years:
    raise ValueError(f'Duplicate years remain after aggregation: {duplicate_years}')

analysis_df['stability_regime'] = pd.qcut(
    analysis_df['sentiment_deviation'],
    q=3,
    labels=['stable', 'neutral', 'unstable'],
    duplicates='drop',
)

analysis_df.head()


In [ ]:
correlation = analysis_df['sentiment_deviation'].corr(analysis_df['next_year_return'])

regime_summary = (
    analysis_df.assign(
        is_negative=analysis_df['next_year_return'] < 0,
        is_above_20=analysis_df['next_year_return'] > 0.20,
    )
    .groupby('stability_regime', observed=False)
    .agg(
        observations=('next_year_return', 'size'),
        mean_return=('next_year_return', 'mean'),
        median_return=('next_year_return', 'median'),
        std_return=('next_year_return', 'std'),
        negative_return_pct=('is_negative', 'mean'),
        above_20pct_return_pct=('is_above_20', 'mean'),
    )
)

regime_order = ['stable', 'neutral', 'unstable']
regime_summary = regime_summary.reindex(regime_order)
regime_summary[['negative_return_pct', 'above_20pct_return_pct']] = (
    regime_summary[['negative_return_pct', 'above_20pct_return_pct']].mul(100)
)

regime_summary['decision_insight'] = pd.Series(
    {
        'stable': 'favorable environment',
        'neutral': 'mixed environment',
        'unstable': 'risk elevated',
    }
)

regime_strategy_map = pd.DataFrame(
    {
        'stability_regime': regime_order,
        'suggested_risk_level': ['high', 'medium', 'low'],
        'suggested_strategy_type': [
            'premium selling / directional',
            'balanced / hedged',
            'defensive / defined risk',
        ],
        'notes': [
            'favorable environment, lower instability',
            'mixed environment',
            'high uncertainty, reduce exposure',
        ],
    }
)

print('correlation (deviation vs forward returns):', round(correlation, 4))
display(regime_summary.round(4))
display(regime_strategy_map)



## Execution Rules and Volatility Confirmation

Translate the stability regimes into executable trading rules and verify whether unstable regimes correspond to larger absolute moves and higher return volatility.


In [ ]:
regime_order = ['stable', 'neutral', 'unstable']

regime_execution_rules = pd.DataFrame(
    {
        'stability_regime': regime_order,
        'position_size_multiplier': [1.25, 1.0, 0.5],
        'max_risk_per_trade': ['higher', 'standard', 'low'],
        'preferred_strategy': [
            'credit spreads / directional trades',
            'balanced spreads',
            'defined-risk / hedged trades',
        ],
        'avoid_strategy': [
            'over-hedging',
            'over-leverage',
            'naked exposure',
        ],
    }
)

regime_volatility_analysis = (
    analysis_df.assign(absolute_return=analysis_df['next_year_return'].abs())
    .groupby('stability_regime', observed=False)
    .agg(
        observations=('next_year_return', 'size'),
        average_absolute_return=('absolute_return', 'mean'),
        return_std_dev=('next_year_return', 'std'),
    )
    .reindex(regime_order)
    .reset_index()
)

unstable_average_absolute_return = regime_volatility_analysis.loc[
    regime_volatility_analysis['stability_regime'] == 'unstable',
    'average_absolute_return',
].iloc[0]
unstable_return_std_dev = regime_volatility_analysis.loc[
    regime_volatility_analysis['stability_regime'] == 'unstable',
    'return_std_dev',
].iloc[0]
peer_max_average_absolute_return = regime_volatility_analysis.loc[
    regime_volatility_analysis['stability_regime'] != 'unstable',
    'average_absolute_return',
].max()
peer_max_return_std_dev = regime_volatility_analysis.loc[
    regime_volatility_analysis['stability_regime'] != 'unstable',
    'return_std_dev',
].max()

unstable_shows_higher_volatility = bool(
    (unstable_average_absolute_return >= peer_max_average_absolute_return)
    and (unstable_return_std_dev >= peer_max_return_std_dev)
)

volatility_confirmation = pd.DataFrame(
    {
        'check': ['unstable regime shows higher volatility'],
        'result': [unstable_shows_higher_volatility],
        'detail': [
            (
                f"unstable avg |return|={unstable_average_absolute_return:.4f}; "
                f"max peer avg |return|={peer_max_average_absolute_return:.4f}; "
                f"unstable std={unstable_return_std_dev:.4f}; "
                f"max peer std={peer_max_return_std_dev:.4f}"
            )
        ],
    }
)

display(regime_execution_rules)
display(regime_volatility_analysis.round(4))
display(volatility_confirmation)


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    analysis_df['sentiment_deviation'],
    analysis_df['next_year_return'].abs(),
    alpha=0.7,
)
plt.title('Sentiment Deviation vs Absolute Forward Returns')
plt.xlabel('Sentiment Deviation')
plt.ylabel('|Next-Year Return|')
plt.tight_layout()
plt.show()


In [ ]:
regime_plot_df = (
    analysis_df.assign(
        regime_level=analysis_df['stability_regime'].map(
            {'stable': 0, 'neutral': 1, 'unstable': 2}
        )
    )
    .sort_values('year', kind='mergesort')
)

regime_colors = {
    'stable': '#d9f2d9',
    'neutral': '#fff3cd',
    'unstable': '#f8d7da',
}

years = regime_plot_df['year'].to_numpy(dtype=float)
if len(years) > 1:
    midpoints = (years[:-1] + years[1:]) / 2
    left_edges = np.concatenate(([years[0] - (midpoints[0] - years[0])], midpoints))
    right_edges = np.concatenate((midpoints, [years[-1] + (years[-1] - midpoints[-1])]))
else:
    left_edges = years - 0.5
    right_edges = years + 0.5

fig, ax1 = plt.subplots(figsize=(11, 5))
for left_edge, right_edge, regime in zip(
    left_edges,
    right_edges,
    regime_plot_df['stability_regime'],
):
    ax1.axvspan(
        left_edge,
        right_edge,
        color=regime_colors[str(regime)],
        alpha=0.25,
        zorder=0,
    )

ax1.step(
    regime_plot_df['year'],
    regime_plot_df['regime_level'],
    where='mid',
    color='tab:blue',
    linewidth=2,
    label='Stability Regime',
)
ax1.set_xlabel('Year')
ax1.set_ylabel('Stability Regime', color='tab:blue')
ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(['stable', 'neutral', 'unstable'])
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(
    regime_plot_df['year'],
    regime_plot_df['next_year_return'],
    color='tab:green',
    marker='o',
    linewidth=2,
    label='Next-Year Return',
)
ax2.axhline(0, color='tab:gray', linestyle='--', linewidth=1)
ax2.set_ylabel('Next-Year Return', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')

lines = ax1.get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='best')
plt.title('Stability Regimes Over Time with Forward Returns')
plt.tight_layout()
plt.show()

regime_plot_df[['year', 'stability_regime', 'next_year_return']].tail()



In [ ]:
execution_plot_df = regime_plot_df.copy()
reduce_size_points = execution_plot_df.loc[
    execution_plot_df['stability_regime'].eq('unstable'),
    ['year', 'next_year_return'],
]

fig, ax1 = plt.subplots(figsize=(11, 5))
for left_edge, right_edge, regime in zip(
    left_edges,
    right_edges,
    execution_plot_df['stability_regime'],
):
    ax1.axvspan(
        left_edge,
        right_edge,
        color=regime_colors[str(regime)],
        alpha=0.25,
        zorder=0,
    )

ax1.step(
    execution_plot_df['year'],
    execution_plot_df['regime_level'],
    where='mid',
    color='tab:blue',
    linewidth=2,
    label='Stability Regime',
)
ax1.set_xlabel('Year')
ax1.set_ylabel('Stability Regime', color='tab:blue')
ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(['stable', 'neutral', 'unstable'])
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(
    execution_plot_df['year'],
    execution_plot_df['next_year_return'],
    color='tab:green',
    marker='o',
    linewidth=2,
    label='Next-Year Return',
)
ax2.axhline(0, color='tab:gray', linestyle='--', linewidth=1)
ax2.set_ylabel('Next-Year Return', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')

if not reduce_size_points.empty:
    ax2.scatter(
        reduce_size_points['year'],
        reduce_size_points['next_year_return'],
        color='tab:red',
        marker='v',
        s=80,
        label='Reduce Position Size',
    )
    first_reduce_year = reduce_size_points['year'].iloc[0]
    first_reduce_return = reduce_size_points['next_year_return'].iloc[0]
    note_offset = max(execution_plot_df['next_year_return'].abs().max() * 0.2, 0.05)
    ax2.annotate(
        'Reduce position size in unstable regimes',
        xy=(first_reduce_year, first_reduce_return),
        xytext=(first_reduce_year, first_reduce_return + note_offset),
        arrowprops={'arrowstyle': '->', 'color': 'tab:red'},
        color='tab:red',
        fontsize=10,
    )

lines = ax1.get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
legend_handles, legend_labels = ax2.get_legend_handles_labels()
ax1.legend(lines + legend_handles, labels + legend_labels, loc='best')
plt.title('Execution Overlay: Stability Regimes, Returns, and Position Sizing')
plt.tight_layout()
plt.show()
